# 07 - The LANL probe: the honest test on real data

The synthetic demos let me choose the task, so I can always build one where a cover
wins. This probe is the attempt to break that circularity on real data: the LANL ARCS
authentication dataset, with labelled red-team logons. The dataset is email-gated and
large, so this notebook runs the actual harness on a small synthetic slice to teach
the method, then reports the committed real-data numbers, which are a pre-registered
null.

The functions below are imported straight from `run_lanl.py`, so what runs here is the
same code that produced the real result.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from sklearn.metrics import average_precision_score
from src.lanl import synthetic_slice, build_windows
from src.baselines import History, BASELINE_NAMES, COVER_NAMES
from run_lanl import assemble, fit_score, recall_at_fpr, sheaf_edge_scores
np.set_printoptions(precision=3, suppress=True)

## The task

Edge-level: given an authentication `src -> dst` in a time window, we ask whether it was red-team
lateral movement. We bin a stream of events into time windows, build a directed access
graph per window, and label each edge against the red-team list. The split is
temporal: the earliest windows train, the rest test. The positive rate is tiny.

In [2]:
# synthetic slice: a fabricated event stream with planted lateral movement (plumbing,
# not evidence). It exercises the exact pipeline the real data runs through.
auths, red, ws = synthetic_slice(seed=0)
windows = list(build_windows(iter(auths), ws))
split_k = int(len(windows) * 0.5)
print(f"windows: {len(windows)}  (first {split_k} train, rest test)")
print(f"edges total: {sum(len(w.edges) for w in windows):,}")

windows: 40  (first 20 train, rest test)
edges total: 8,671


## The honest baseline: novelty + degree

A real authentication graph is essentially one giant connected component, so counting
components or measuring global reachability, the trick that wins on the synthetic
expressivity task, is uninformative here. What works on LANL is mundane: whether this
`src -> dst` edge, or this credential on this host, has been seen before. Red-team movement
so often lands on a host a credential has never touched. That novelty-plus-degree
detector, with no GNN, is the bar to beat. The cover block is a bounded,
ego-net-restricted reachability feature; we ask whether it adds anything over the bar.

In [3]:
print("baseline features:", BASELINE_NAMES)
print("cover features:   ", COVER_NAMES)

baseline features: ['novel_edge', 'novel_cred', 'auth_failed', 'src_out_window', 'dst_in_window', 'log_src_out_global', 'log_dst_in_global']
cover features:    ['src_reach_K', 'dst_reach_K', 'dst_in_reach_K']


## Two things that make novelty meaningful

`assemble` walks the windows through one continuous history, so an edge in a test
window is judged "seen before" against everything earlier, never reset at the split.
On real data a warm-up also seeds that history from the days before the window
(`--warmup-days`), so the first window is not flooded with spurious novel edges. Here
the history starts empty and accumulates as we go.

In [4]:
history = History()
Xb, Xc, y, is_train, n_te = assemble(windows, red, history, split_k)
ytr, yte = y[is_train], y[~is_train]
print(f"test edges: {len(yte):,}   positives: {int(yte.sum())} ({yte.mean():.3%})")

def scores_pr(Xtr, Xte):
    s = fit_score(Xtr, ytr, Xte)
    return average_precision_score(yte, s), recall_at_fpr(yte, s)

pr_b, rc_b = scores_pr(Xb[is_train], Xb[~is_train])
Xall = np.hstack([Xb, Xc])
pr_c, rc_c = scores_pr(Xall[is_train], Xall[~is_train])
sh = sheaf_edge_scores(windows, split_k, red)[~is_train]
pr_s, rc_s = average_precision_score(yte, sh), recall_at_fpr(yte, sh)

print(f"\n{'model':24s} {'PR-AUC':>8s} {'recall@0.1%FPR':>14s}")
print("-" * 48)
print(f"{'baseline (novelty+deg)':24s} {pr_b:>8.3f} {rc_b:>14.2f}")
print(f"{'baseline + cover':24s} {pr_c:>8.3f} {rc_c:>14.2f}")
print(f"{'sheaf NN (end-to-end)':24s} {pr_s:>8.3f} {rc_s:>14.2f}")

test edges: 4,304   positives: 32 (0.743%)



model                      PR-AUC recall@0.1%FPR
------------------------------------------------
baseline (novelty+deg)      0.970           0.97
baseline + cover            0.970           0.97
sheaf NN (end-to-end)       0.007           0.00


On the synthetic slice the attacks are novel edges, so the novelty baseline already
catches them and neither the cover nor the sheaf NN adds anything. That is the same
shape of result we get on the real data.

## The real-data result (committed, two red-team windows)

On LANL the campaign is single-pivot (host `C17693`), so this tests temporal
robustness. With logistic regression on standardised features, across both windows the
no-GNN novelty baseline wins. Neither the reachability cover nor a trained sheaf NN
beats it.

Day 8 (28 train positives, 246 test):

| model | PR-AUC | recall@0.1%FPR |
|---|---|---|
| baseline (novelty+degree) | 0.044 | 0.68 |
| baseline + cover | 0.029 | 0.58 |
| sheaf NN (end-to-end) | 0.030 | 0.50 |

Day 12 (129 train positives, 80 test):

| model | PR-AUC | recall@0.1%FPR |
|---|---|---|
| baseline (novelty+degree) | 0.011 | 0.55 |
| baseline + cover | 0.000 | 0.01 |
| sheaf NN (end-to-end) | 0.004 | 0.20 |

An earlier draft showed a day-8 cover lift (PR-AUC 0.16); that was a conditioning
artifact of an unconverged fit on unscaled features, and it vanished once the features
were standardised. The full story, with caveats (the unusable ~3,900 alerts/day
operating point, the single-pivot campaign), is in
[../docs/lanl-probe.md](../docs/lanl-probe.md).

## Reading

The pre-registered prediction was that the novelty baseline would be hard to beat, and
it held: on real lateral-movement data, where the signal is not handed over by
construction, neither a cover nor a trained sheaf NN beats a hash set of seen edges.
The dominant signal here is novelty, which is not a graph-structural property at all.
That is the honest null, and reporting it is the point.

What survives regardless: the harness is real and reusable. Novelty with warm-up and
continuous history, a sparse `reachable_counts` that made ~4,000-node-per-window graphs
tractable, and a sparse end-to-end sheaf edge detector.